# 2. Process South Australia (SA) battery energy storage system (BESS) Data

## Purpose
This notebook regenerates the 100-household, one-year BESS aggregate from the selected signal definition.

## Run Flow
1. Setup
2. Select Profile And Households
3. Export Aggregate And Datasets
4. QA Snapshot

## Setup

Load workflow helpers and either reuse diagnostic outputs or rerun diagnostics if needed.

In [1]:
from pathlib import Path
import sys
import pandas as pd


def find_publication_project(start: Path) -> Path:
    """Find publication/journal_article_1 from this moved notebook folder."""
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "scripts" / "sa_bess_publication_workflow.py").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")


PROJECT_DIR = find_publication_project(Path.cwd())
sys.path.insert(0, str(PROJECT_DIR / "scripts"))
from sa_bess_publication_workflow import run_diagnostics_and_select, process_selected_sample, PROCESSED_DIR

print(f"Publication project: {PROJECT_DIR}")

Publication project: <local path redacted>


## Select Profile And Households

The diagnostics function writes the report and returns the selected profile, selected households, and selected AEST window.

In [2]:
selected_profile_id, selected_households, selection_info, metric_summary = run_diagnostics_and_select()
selected_households.head()

[diagnostics] processed 1/547 parquet files


[diagnostics] processed 25/547 parquet files


[diagnostics] processed 50/547 parquet files


[diagnostics] processed 75/547 parquet files


[diagnostics] processed 100/547 parquet files


...

[diagnostics] processed 500/547 parquet files


[diagnostics] processed 525/547 parquet files


[diagnostics] processed 547/547 parquet files


[diagnostics] selected profile: current_polarity_adjusted
[diagnostics] report written: <local path redacted>


,site_id,selected_overlap_observed_timestamps_pre_fill,selected_overlap_meaningful_negative_count_pre_fill,selected_overlap_min_underlying_load_kW_pre_fill,selected_overlap_missingness_pct_pre_fill,first_observed_date,has_meaningful_negative_underlying_load_pre_fill,selection_rank,is_selected_sample,state,postcode,latitude,longitude,dc_capacity_kw,ac_capacity_kw,monitoring_start
0,1995273389,105109.0,0.0,0.007010,0.010464,2024-01-01,False,1,1,SA,5090,-34.830,138.7,5.04,5.0,2020-04-20
1,245185730,105105.0,0.0,0.078340,0.014269,2024-01-26,False,2,1,VIC,3453,-36.900,144.2,11.84,8.2,2022-03-24
2,905921552,105102.0,0.0,0.148773,0.017123,2024-01-01,False,3,1,NSW,2167,-33.975,150.9,13.26,10.0,2022-09-20
3,1805446999,105097.0,0.0,0.168707,0.021880,2024-01-01,False,4,1,VIC,3040,-37.755,144.9,13.30,10.0,2021-12-09
4,1284666164,105095.0,0.0,0.209110,0.023782,2024-01-01,False,5,1,VIC,3072,-37.755,145.0,5.12,6.0,2023-02-12


## Export Aggregate And Datasets

The 5-minute aggregate sums household power. The 30-minute PyNNLF datasets use arithmetic mean power.

In [3]:
aggregate_5min, site_timeseries, household_summary = process_selected_sample(selected_profile_id, selected_households, selection_info)
print(aggregate_5min.shape)
print(PROCESSED_DIR)

[selected-source] processed 1/547 parquet files


[selected-source] processed 25/547 parquet files


[selected-source] processed 50/547 parquet files


[selected-source] processed 75/547 parquet files


[selected-source] processed 100/547 parquet files


...

[fill] processed 70/100 selected households


[fill] processed 80/100 selected households


[fill] processed 90/100 selected households


[fill] processed 100/100 selected households


(105120, 4)
<local path redacted>


## QA Snapshot

Check row counts, timestamp spacing, and selected household count.

In [4]:
print('5-min rows:', len(aggregate_5min))
print('households:', household_summary['site_id'].nunique())
print('start:', aggregate_5min['datetime'].min())
print('end:', aggregate_5min['datetime'].max())
print('missing cells:', aggregate_5min.isna().sum().sum())

5-min rows: 105120
households: 100
start: 2024-02-26 00:00:00
end: 2025-02-24 23:55:00
missing cells: 0
